## NLP y gráficas

In [ ]:
# ==========================================
# NOTEBOOK: 03_nlp_sesgo_ofertas.ipynb
# OBJETIVO:
# Clasificar ofertas de empleo tech según su nivel de sesgo:
# bajo, medio o alto
# ==========================================

# ==========
# 1. LIBRERÍAS
# ==========
import os
import re
import pandas as pd
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ==========
# 2. DETECTAR RUTA BASE
# ==========
print("Carpeta actual:", os.getcwd())

if Path("data/processed").exists():
    BASE_PATH = Path(".")
elif Path("../data/processed").exists():
    BASE_PATH = Path("..")
else:
    raise FileNotFoundError("No se encontró la carpeta data/processed")

PROCESSED_PATH = BASE_PATH / "data" / "processed"

# ==========
# 3. CARGAR DATASET LIMPIO
# ==========
file_path = PROCESSED_PATH / "job_offers_bias_clean.csv"
df = pd.read_csv(file_path)

print("\nShape del dataset:", df.shape)
display(df.head())

# ==========
# 4. REVISIÓN BÁSICA
# ==========
print("\nColumnas:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.dtypes)

print("\nValores nulos:")
print(df.isnull().sum())

# ==========
# 5. LIMPIEZA FINAL DEL TEXTO
# Aunque ya viene limpio, hacemos una normalización sencilla
# ==========
def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

df["descripcion"] = df["descripcion"].astype(str).apply(clean_text)
df["nivel_sesgo"] = df["nivel_sesgo"].astype(str).str.lower().str.strip()

# Eliminar posibles filas vacías
df = df.dropna(subset=["descripcion", "nivel_sesgo"]).reset_index(drop=True)
df = df[df["descripcion"].str.len() > 0].reset_index(drop=True)

print("\nDistribución de clases:")
print(df["nivel_sesgo"].value_counts())

# ==========
# 6. GRÁFICA 1: DISTRIBUCIÓN DE CLASES
# ==========
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="nivel_sesgo", order=df["nivel_sesgo"].value_counts().index, hue="nivel_sesgo", dodge=False, legend=False)
plt.title("Distribución del nivel de sesgo")
plt.xlabel("Nivel de sesgo")
plt.ylabel("Número de ofertas")
plt.show()

# ==========
# 7. PREPARAR DATOS
# X = texto
# y = nivel_sesgo
# ==========
X = df["descripcion"]
y = df["nivel_sesgo"]

# ==========
# 8. DIVIDIR TRAIN / TEST
# ==========
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTamaño train:", X_train.shape)
print("Tamaño test:", X_test.shape)

# ==========
# 9. MODELO NLP
# TF-IDF + Logistic Regression
# Es una opción rápida, clara y muy útil para un proyecto académico.
# ==========
nlp_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words=None,
        ngram_range=(1, 2),
        max_features=1000
    )),
    ("clf", LogisticRegression(
        max_iter=200,
        random_state=42
    ))
])

# ==========
# 10. ENTRENAR
# ==========
nlp_model.fit(X_train, y_train)

# ==========
# 11. PREDECIR
# ==========
y_pred = nlp_model.predict(X_test)

# ==========
# 12. MÉTRICAS
# ==========
accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy del modelo NLP:", round(accuracy, 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ==========
# 13. MATRIZ DE CONFUSIÓN
# ==========
labels = sorted(df["nivel_sesgo"].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

print("\nMatriz de confusión:")
print(cm)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", xticklabels=labels, yticklabels=labels)
plt.title("Matriz de confusión - NLP sesgo en ofertas")
plt.xlabel("Predicción")
plt.ylabel("Valor real")
plt.show()

# ==========
# 14. LONGITUD DEL TEXTO
# Añadimos una métrica simple para visualizar diferencias
# ==========
df["longitud_texto"] = df["descripcion"].apply(lambda x: len(x.split()))

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="nivel_sesgo", y="longitud_texto", hue="nivel_sesgo", dodge=False, legend=False)
plt.title("Longitud del texto por nivel de sesgo")
plt.xlabel("Nivel de sesgo")
plt.ylabel("Número de palabras")
plt.show()

# ==========
# 15. FRECUENCIA DE PALABRAS CLAVE SESGADAS
# Revisamos algunas palabras asociadas a lenguaje más agresivo/competitivo
# ==========
biased_words = [
    "agresivo", "dominante", "competitivo", "fuerte",
    "ambicioso", "guerrero", "rockstar", "ninja",
    "killer", "brillante", "duro", "imparable"
]

def count_biased_words(text):
    words = text.split()
    return sum(1 for word in words if word in biased_words)

df["conteo_palabras_sesgadas"] = df["descripcion"].apply(count_biased_words)

plt.figure(figsize=(8, 5))
sns.barplot(
    data=df.groupby("nivel_sesgo")["conteo_palabras_sesgadas"].mean().reset_index(),
    x="nivel_sesgo",
    y="conteo_palabras_sesgadas",
    hue="nivel_sesgo",
    dodge=False,
    legend=False
)
plt.title("Promedio de palabras sesgadas por nivel")
plt.xlabel("Nivel de sesgo")
plt.ylabel("Promedio de palabras sesgadas")
plt.show()

# ==========
# 16. EJEMPLOS DE PREDICCIÓN
# ==========
examples = pd.DataFrame({
    "descripcion": [
        "buscamos una persona colaborativa con ganas de aprender y trabajar en equipo",
        "necesitamos una persona competitiva y orientada a resultados para un entorno exigente",
        "queremos un rockstar del codigo con mentalidad ganadora y agresiva"
    ]
})

examples["prediccion"] = nlp_model.predict(examples["descripcion"])

print("\nEjemplos de predicción:")
display(examples)

# ==========
# 17. TABLA DE RESULTADOS REALES VS PREDICHOS
# ==========
results = pd.DataFrame({
    "texto": X_test.values,
    "real": y_test.values,
    "predicho": y_pred
})

print("\nMuestra de resultados:")
display(results.head(10))
